# Scaling-law fitting experiments

This notebook only runs the fitted total-loss experiment. It does not detect
transitions, scan momentum values, or draw phase markers. The corrected
theoretical shapes are

$$
\widehat L_{\rm Polyak}(k)
= C_T\rho^k+C_S(\eta_\rho k)^{-s}
+C_N\frac{\sigma^2}{B}\eta_\rho,
$$

$$
\widehat L_{\rm Nesterov}(k)
= \widetilde C_T\min\{1,(\eta k)^{-s}\}\rho^k
+\widetilde C_S(\eta_\rho k)^{-s}
+\widetilde C_N\frac{\sigma^2}{B}
\min\{\eta_\rho,\eta_\rho^{1/\beta}\},
$$

where $\eta_\rho=\eta/(1-\rho)$.



In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.lines import Line2D


# =============================================================================
# 1. Device and output directory
# =============================================================================

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.float64
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float32
else:
    raise RuntimeError("A CUDA or Apple MPS GPU is required for the simulations.")

OUTPUT_DIR = Path("loss/results/generated/scaling_law_fits")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE}, dtype={DTYPE}, torch={torch.__version__}")
print(f"output directory: {OUTPUT_DIR.resolve()}")




## Experimental settings

All six $(s,\beta)$ pairs use the same batch size, learning rate, momentum,
label-noise level, dimension, and number of independent runs. Existing data
files are reused unless `OVERWRITE` is set to `True`.



In [ ]:
# =============================================================================
# 2. Shared experiment configuration
# =============================================================================

S_VALUES = (0.3, 0.5)
BETA_VALUES = (2.0, 4.0, 6.0)

# Rows of the final figure correspond to s; columns correspond to beta.
SETTINGS = tuple(
    (s, beta)
    for s in S_VALUES
    for beta in BETA_VALUES
)

BATCH_SIZE = 64
ETA = 0.06
SIGMA = 0.2
DIMENSION = 4096
RUNS = 100
TRAIN_STEPS = 500_000
N_CHECKPOINTS = 1_200
SEED = 7
ONE_MINUS_RHO = 0.03
RHO = 1.0 - ONE_MINUS_RHO

ALGORITHMS = (
    "polyak",
    "nesterov",
)

COLORS = {
    "polyak": "C1",
    "nesterov": "C2",
}

# False makes the notebook resume safely from already completed settings.
OVERWRITE = False

CHECKPOINTS = np.unique(
    np.concatenate((
        np.asarray([0], dtype=int),
        np.geomspace(
            1,
            TRAIN_STEPS,
            N_CHECKPOINTS,
        ).round().astype(int),
    ))
).astype(int)

print(f"number of settings: {len(SETTINGS)}")
print(f"settings: {SETTINGS}")
print(f"training steps per run: {TRAIN_STEPS:,}")




In [ ]:
# =============================================================================
# 3. File names
# =============================================================================

def number_tag(value: float) -> str:
    """Convert 0.3 to 0_3 and 2.0 to 2 for file names."""

    return format(float(value), ".15g").replace(".", "_")


def data_path_for(s: float, beta: float) -> Path:
    return OUTPUT_DIR / (
        f"scaling_law_s{number_tag(s)}_beta{number_tag(beta)}_data.npz"
    )


def figure_stem_for(s: float, beta: float) -> Path:
    return OUTPUT_DIR / (
        f"scaling_law_s{number_tag(s)}_beta{number_tag(beta)}"
    )




## PLK simulation

The empirical curve is the exact population excess risk averaged over 100
independent GPU replicas. A fresh curvature-aligned Gaussian gradient-noise
realization is generated at every step.



In [ ]:
# =============================================================================
# 4. PLK simulation
# =============================================================================

@torch.no_grad()
def simulate_empirical_curve(
    algorithm: str,
    *,
    s: float,
    beta: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Simulate one empirical mean-loss trajectory."""

    if algorithm not in ALGORITHMS:
        raise ValueError(f"unknown algorithm: {algorithm}")

    torch.manual_seed(seed)

    j = torch.arange(
        1,
        DIMENSION + 1,
        device=DEVICE,
        dtype=DTYPE,
    )

    eigenvalues = j.pow(-beta)
    sqrt_eigenvalues = eigenvalues.sqrt()

    theta_star = j.pow(
        -0.5 * (1.0 + (s - 1.0) * beta)
    )

    current = -theta_star.expand(
        RUNS,
        -1,
    ).clone()

    previous = current.clone()

    checkpoint_set = set(CHECKPOINTS.tolist())

    saved_steps = [0]

    initial_loss = 0.5 * (
        current.square() * eigenvalues
    ).sum(dim=1)

    saved_loss = [
        float(initial_loss.mean().item())
    ]

    for step in range(1, TRAIN_STEPS + 1):

        momentum = current - previous

        if algorithm == "nesterov":
            evaluation = current + RHO * momentum
        else:
            evaluation = current

        evaluation_loss = 0.5 * (
            evaluation.square() * eigenvalues
        ).sum(dim=1)

        noise_scale = (
            (evaluation_loss + SIGMA**2) / BATCH_SIZE
        ).sqrt().unsqueeze(1)

        gradient = (
            evaluation * eigenvalues
            + torch.randn_like(evaluation)
            * sqrt_eigenvalues
            * noise_scale
        )

        new = (
            current
            + RHO * momentum
            - ETA * gradient
        )

        previous, current = current, new

        if step in checkpoint_set:

            loss = 0.5 * (
                current.square() * eigenvalues
            ).sum(dim=1)

            saved_steps.append(step)
            saved_loss.append(
                float(loss.mean().item())
            )

        if step % 50_000 == 0:
            print(
                f"  (s, beta)=({s:g}, {beta:g}), "
                f"{algorithm}: {step:,}/{TRAIN_STEPS:,}",
                flush=True,
            )

    return (
        np.asarray(saved_steps, dtype=float),
        np.asarray(saved_loss, dtype=float),
    )




## Corrected scaling-law fit

Each theoretical term keeps its predicted shape. Three nonnegative
prefactors are fitted by relative least squares.



In [ ]:
# =============================================================================
# 5. Corrected theoretical shapes and nonnegative fit
# =============================================================================

def theory_matrix_at_steps(
    steps: np.ndarray,
    algorithm: str,
    *,
    s: float,
    beta: float,
) -> np.ndarray:
    """Return transient, signal, and noise shapes at the requested steps."""

    k = np.maximum(
        np.asarray(steps, dtype=float),
        1.0,
    )

    eta_rho = ETA / ONE_MINUS_RHO

    signal = (
        eta_rho * k
    ) ** (-s)

    if algorithm == "polyak":

        transient = RHO**k

        noise_level = (
            SIGMA**2
            / BATCH_SIZE
            * eta_rho
        )

    elif algorithm == "nesterov":

        transient = (
            np.minimum(
                1.0,
                (ETA * k) ** (-s),
            )
            * RHO**k
        )

        noise_level = (
            SIGMA**2
            / BATCH_SIZE
            * min(
                eta_rho,
                eta_rho ** (1.0 / beta),
            )
        )

    else:
        raise ValueError(f"unknown algorithm: {algorithm}")

    return np.column_stack((
        transient,
        signal,
        np.full_like(k, noise_level),
    ))


def nonnegative_lstsq(
    matrix: np.ndarray,
    target: np.ndarray,
) -> np.ndarray:
    """Solve the three-variable nonnegative least-squares problem exactly."""

    number_of_terms = matrix.shape[1]

    best_coefficient = np.zeros(
        number_of_terms,
        dtype=float,
    )

    best_error = np.inf

    for mask in range(1, 1 << number_of_terms):

        active = [
            index
            for index in range(number_of_terms)
            if mask & (1 << index)
        ]

        active_coefficient, *_ = np.linalg.lstsq(
            matrix[:, active],
            target,
            rcond=None,
        )

        if np.any(active_coefficient < 0):
            continue

        coefficient = np.zeros(
            number_of_terms,
            dtype=float,
        )

        coefficient[active] = active_coefficient

        error = np.mean(
            (matrix @ coefficient - target) ** 2
        )

        if error < best_error:
            best_coefficient = coefficient
            best_error = error

    return best_coefficient


def fit_empirical_curve(
    steps: np.ndarray,
    empirical_loss: np.ndarray,
    algorithm: str,
    *,
    s: float,
    beta: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Fit C_T, C_S, and C_N and return the fitted total loss."""

    terms = theory_matrix_at_steps(
        steps,
        algorithm,
        s=s,
        beta=beta,
    )

    eta_rho = ETA / ONE_MINUS_RHO

    valid = (
        (steps > 0)
        & np.isfinite(empirical_loss)
        & (empirical_loss > 0)
        & (eta_rho * steps >= 1.0)
    )

    scale = np.maximum(
        empirical_loss[valid],
        np.finfo(float).tiny,
    )

    coefficient = nonnegative_lstsq(
        terms[valid] / scale[:, None],
        np.ones(valid.sum(), dtype=float),
    )

    fitted_loss = terms @ coefficient

    return coefficient, fitted_loss




In [ ]:
# =============================================================================
# 6. Save and load one completed setting
# =============================================================================

def save_setting_data(
    path: Path,
    *,
    s: float,
    beta: float,
    results: dict[str, dict[str, np.ndarray]],
) -> None:
    """Save only the empirical curves, fitted curves, and fitted coefficients."""

    np.savez_compressed(
        path,
        s=np.asarray(s, dtype=float),
        beta=np.asarray(beta, dtype=float),
        batch_size=np.asarray(BATCH_SIZE, dtype=int),
        eta=np.asarray(ETA, dtype=float),
        sigma=np.asarray(SIGMA, dtype=float),
        rho=np.asarray(RHO, dtype=float),
        one_minus_rho=np.asarray(ONE_MINUS_RHO, dtype=float),
        dimension=np.asarray(DIMENSION, dtype=int),
        runs=np.asarray(RUNS, dtype=int),
        train_steps=np.asarray(TRAIN_STEPS, dtype=int),
        polyak_steps=np.asarray(
            results["polyak"]["steps"],
            dtype=float,
        ),
        polyak_original_loss=np.asarray(
            results["polyak"]["empirical"],
            dtype=float,
        ),
        polyak_fitted_loss=np.asarray(
            results["polyak"]["prediction"],
            dtype=float,
        ),
        polyak_fitted_coefficients=np.asarray(
            results["polyak"]["coefficient"],
            dtype=float,
        ),
        nesterov_steps=np.asarray(
            results["nesterov"]["steps"],
            dtype=float,
        ),
        nesterov_original_loss=np.asarray(
            results["nesterov"]["empirical"],
            dtype=float,
        ),
        nesterov_fitted_loss=np.asarray(
            results["nesterov"]["prediction"],
            dtype=float,
        ),
        nesterov_fitted_coefficients=np.asarray(
            results["nesterov"]["coefficient"],
            dtype=float,
        ),
    )


def load_setting_data(
    path: Path,
) -> dict[str, object]:
    """Load one saved setting into the plotting structure."""

    with np.load(path, allow_pickle=False) as data:

        return {
            "s": float(data["s"]),
            "beta": float(data["beta"]),
            "polyak": {
                "steps": np.asarray(
                    data["polyak_steps"],
                    dtype=float,
                ),
                "empirical": np.asarray(
                    data["polyak_original_loss"],
                    dtype=float,
                ),
                "prediction": np.asarray(
                    data["polyak_fitted_loss"],
                    dtype=float,
                ),
                "coefficient": np.asarray(
                    data["polyak_fitted_coefficients"],
                    dtype=float,
                ),
            },
            "nesterov": {
                "steps": np.asarray(
                    data["nesterov_steps"],
                    dtype=float,
                ),
                "empirical": np.asarray(
                    data["nesterov_original_loss"],
                    dtype=float,
                ),
                "prediction": np.asarray(
                    data["nesterov_fitted_loss"],
                    dtype=float,
                ),
                "coefficient": np.asarray(
                    data["nesterov_fitted_coefficients"],
                    dtype=float,
                ),
            },
        }




In [ ]:
# =============================================================================
# 7. Run or resume all settings
# =============================================================================

def run_setting(
    s: float,
    beta: float,
    *,
    setting_index: int,
) -> dict[str, object]:
    """Run, fit, save, and return one (s, beta) setting."""

    output_path = data_path_for(s, beta)

    if output_path.exists() and not OVERWRITE:
        print(f"loading existing data: {output_path}")
        return load_setting_data(output_path)

    print("=" * 72)
    print(f"running (s, beta)=({s:g}, {beta:g})")

    results: dict[str, dict[str, np.ndarray]] = {}

    for algorithm_index, algorithm in enumerate(ALGORITHMS):

        simulation_seed = (
            SEED
            + 100_000 * setting_index
            + 10_000 * algorithm_index
        )

        steps, empirical_loss = simulate_empirical_curve(
            algorithm,
            s=s,
            beta=beta,
            seed=simulation_seed,
        )

        coefficient, fitted_loss = fit_empirical_curve(
            steps,
            empirical_loss,
            algorithm,
            s=s,
            beta=beta,
        )

        results[algorithm] = {
            "steps": steps,
            "empirical": empirical_loss,
            "prediction": fitted_loss,
            "coefficient": coefficient,
        }

        print(
            algorithm,
            dict(
                zip(
                    ("C_T", "C_S", "C_N"),
                    coefficient,
                )
            ),
        )

    save_setting_data(
        output_path,
        s=s,
        beta=beta,
        results=results,
    )

    print(f"saved: {output_path}")

    loaded = load_setting_data(output_path)

    del results
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return loaded


all_results = {
    (s, beta): run_setting(
        s,
        beta,
        setting_index=setting_index,
    )
    for setting_index, (s, beta) in enumerate(SETTINGS)
}




## Figures

Light curves are empirical mean losses and dark curves are the corresponding
fitted scaling-law predictions. No phase labels or transition markers are
drawn.



In [ ]:
# =============================================================================
# 8. Plot configuration
# =============================================================================

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.figsize": (5, 4),
    "legend.fontsize": 10,
    "legend.handlelength": 1,
})


def algorithm_legend_handles() -> list[Line2D]:
    return [
        Line2D(
            [0],
            [0],
            color=COLORS[algorithm],
            lw=2.5,
            label=algorithm.capitalize(),
        )
        for algorithm in ALGORITHMS
    ]


def plot_setting_on_axis(
    axis: plt.Axes,
    data: dict[str, object],
    *,
    title: str,
    show_xlabel: bool = True,
    show_ylabel: bool = True,
    show_legend: bool = True,
) -> None:
    """Plot the empirical and fitted total-loss curves for one setting."""

    for algorithm in ALGORITHMS:

        result = data[algorithm]

        steps = np.asarray(
            result["steps"],
            dtype=float,
        )

        empirical_loss = np.asarray(
            result["empirical"],
            dtype=float,
        )

        fitted_loss = np.asarray(
            result["prediction"],
            dtype=float,
        )

        positive = (
            (steps > 0)
            & np.isfinite(empirical_loss)
            & np.isfinite(fitted_loss)
            & (empirical_loss > 0)
            & (fitted_loss > 0)
        )

        axis.loglog(
            steps[positive],
            empirical_loss[positive],
            color=COLORS[algorithm],
            alpha=0.32,
            lw=1.4,
        )

        axis.loglog(
            steps[positive],
            fitted_loss[positive],
            color=COLORS[algorithm],
            lw=2.5,
        )

    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_ylim(top=2)
    axis.set_title(title)

    if show_xlabel:
        axis.set_xlabel("Step")

    if show_ylabel:
        axis.set_ylabel("Loss")

    if show_legend:
        axis.legend(
            handles=algorithm_legend_handles(),
            loc="best",
        )




In [ ]:
# =============================================================================
# 9. Save one standalone figure for every setting
# =============================================================================

for s, beta in SETTINGS:

    data = all_results[(s, beta)]

    fig, axis = plt.subplots(figsize=(5, 4))

    plot_setting_on_axis(
        axis,
        data,
        title=rf"$(s,\beta)=({s:g},{beta:g})$",
    )

    fig.tight_layout()

    output_stem = figure_stem_for(s, beta)

    fig.savefig(
        output_stem.with_suffix(".pdf"),
        bbox_inches="tight",
    )

    fig.savefig(
        output_stem.with_suffix(".png"),
        bbox_inches="tight",
        dpi=200,
    )

    plt.show()




In [ ]:
# =============================================================================
# 10. Save the 2 x 3 summary figure
# =============================================================================

fig, axes = plt.subplots(
    len(S_VALUES),
    len(BETA_VALUES),
    figsize=(12, 7),
)

for row_index, s in enumerate(S_VALUES):

    for column_index, beta in enumerate(BETA_VALUES):

        axis = axes[row_index, column_index]

        plot_setting_on_axis(
            axis,
            all_results[(s, beta)],
            title=rf"$(s,\beta)=({s:g},{beta:g})$",
            show_xlabel=(row_index == len(S_VALUES) - 1),
            show_ylabel=(column_index == 0),
            show_legend=False,
        )

fig.legend(
    handles=algorithm_legend_handles(),
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
)

fig.tight_layout(
    rect=(0, 0, 1, 0.96),
)

fig.savefig(
    OUTPUT_DIR / "scaling_law_fits_2x3.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_DIR / "scaling_law_fits_2x3.png",
    bbox_inches="tight",
    dpi=200,
)

plt.show()

print("Saved all data and figures to:")
print(OUTPUT_DIR.resolve())
